# Step 7 - Train final stacking ensemble + MLflow tracking


In [ ]:
# Imports y configuracion del proyecto
from project_config import init_notebook
config = init_notebook()

import utils
logger = utils.init_logger('info')


In [ ]:
import json
import mlflow
import pandas as pd
from pathlib import Path
from sklearn.metrics import average_precision_score, roc_auc_score

from analytics import df_split, threshold_iter, choose_threshold_by_strategy
from models.stacking import (
    StackingEnsemble, MetaLearner,
    LGBMBaseLearner, XGBBaseLearner, CatBoostBaseLearner, RFBaseLearner,
)

mlflow.set_tracking_uri(config['mlflow']['tracking_uri'])
mlflow.set_experiment(config['mlflow']['experiment_name'])

processed = Path(config['project_folder']) / config['data']['processed']['local_path']
train = pd.read_parquet(processed / 'train.parquet')
val = pd.read_parquet(processed / 'val.parquet')
test = pd.read_parquet(processed / 'test.parquet')
selected = json.loads((processed / 'selected_features.json').read_text())

target = config['model']['objective_column']
X_tr, y_tr = df_split(train[selected + [target] + ['date']], target)
X_va, y_va = df_split(val[selected + [target]], target)
X_te, y_te = df_split(test[selected + [target]], target)
dates_tr = X_tr.pop('date')


## 7.1 Entrenar ensemble con MLflow


In [ ]:
with mlflow.start_run(run_name='stacking-v1') as run:
    mlflow.log_params({
        'n_features': len(selected),
        'threshold_strategy': config['model']['threshold_strategy'],
        'class_weight': config['model']['class_weight'],
    })

    ensemble = StackingEnsemble(
        base_learners=[LGBMBaseLearner(), XGBBaseLearner(), CatBoostBaseLearner(), RFBaseLearner()],
        meta_learner=MetaLearner(),
        n_folds=4,
    )
    ensemble.fit(X_tr, y_tr, dates_tr, X_val=X_va, y_val=y_va)

    val_proba = ensemble.predict_proba(X_va)[:, 1]
    test_proba = ensemble.predict_proba(X_te)[:, 1]

    val_pr = average_precision_score(y_va, val_proba)
    test_pr = average_precision_score(y_te, test_proba)
    mlflow.log_metrics({
        'val_pr_auc': val_pr, 'val_roc_auc': roc_auc_score(y_va, val_proba),
        'test_pr_auc': test_pr, 'test_roc_auc': roc_auc_score(y_te, test_proba),
    })

    # Threshold optimo en val
    iters = threshold_iter(y_va, val_proba, min_value=0.05, max_value=0.95, step=0.05)
    best = choose_threshold_by_strategy(
        iters, strategy=config['model']['threshold_strategy'],
        min_precision=config['model']['min_precision'],
        min_recall=config['model']['min_recall'],
    )
    mlflow.log_metric('selected_threshold', best['threshold'])
    print('val PR-AUC:', val_pr, 'test PR-AUC:', test_pr, 'threshold:', best['threshold'])

    import joblib
    artifact = Path(config['project_folder']) / 'models_artifacts' / 'stacking_ensemble.joblib'
    artifact.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(ensemble, artifact)
    mlflow.log_artifact(str(artifact))
